In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/workspace/default/my_volume/BigMart Sales.csv")
)

In [0]:

df = (df
        .dropDuplicates()
        .fillna({"Item_Weight":0, "Outlet_Size": "Unknown"})
        .withColumn("Item_Fat_Content",
                     when(col("Item_Fat_Content").isin("LF", "low fat"),"Low Fat")
                    .when(col("Item_Fat_Content").isin("reg"), "Regular")
                    .otherwise(col("Item_Fat_Content"))
                    )
        
    )

print("Cleaned Rows are:", df.count())
df.display()

**  3. For each Outlet_Identifier, calculates:
Total Sales,
Average Sales,
Number of Unique Items Sold,
Total Sales from High MRP items only (Item_MRP >= 150) 
**

In [0]:
outlet_Summary = (
    df.groupBy("Outlet_Identifier")
    .agg(
        round(sum("Item_Outlet_Sales"), 2).alias("Total Sales"),
        round(avg("Item_Outlet_Sales"), 2).alias("Average Sales"),
        countDistinct("Item_Identifier").alias("Unique Items Sold"),
        round(sum(when(col("Item_MRP")>=150,col("Item_Outlet_Sales")).otherwise(0)), 2).alias("High MRP Sales")

    )
)
outlet_Summary.display()

**STEP 3: Rank outlets by Total Sales (dense rank)**


In [0]:
rank_window = Window.orderBy(desc("Total Sales"))
outlet_ranked = outlet_Summary.withColumn("Outlet Rank", dense_rank().over(rank_window))
outlet_ranked.display()

**Top Selling Items By Outlet**

In [0]:

item_outlet_agg = (
    df.groupBy("Outlet_Identifier", "Item_Type")
    .agg(round(sum("Item_Outlet_Sales"), 2).alias("Item Type Sales"))
)

top_item_window = Window.partitionBy("Outlet_Identifier").orderBy(desc("Item Type Sales"))
top_item_per_outlet = (item_outlet_agg
    .withColumn("Item_Rank", row_number().over(top_item_window))
    .filter(col("Item_Rank") == 1)
    .select("Outlet_Identifier", "Item_Type", "Item Type Sales")
    .withColumnRenamed("Item_Type", "Top_Item_Type")
    .withColumnRenamed("Item Type Sales", "Top_Item_Sales")
)
print("STEP 4 - Top-selling Item_Type per outlet identified")
top_item_per_outlet.display()


**Join**

In [0]:
Final_report = (
    outlet_ranked.join(top_item_per_outlet, on="Outlet_Identifier", how="left")
    .orderBy("Outlet Rank")
     .select(
        "Outlet Rank", "Outlet_Identifier", "Total Sales", "Average Sales",
        "Unique Items Sold", "High MRP Sales", "Top_Item_Type", "Top_Item_Sales"
    )
)

Final_report.display()


**Total Sales percentage**

In [0]:
final_report = (Final_report
                .withColumn("Item_Sales_Percentage", 
                    round(col("Top_Item_Sales")/ col("Total Sales")* 100, 2)
                    )
)
final_report.display()
df.display()

In [0]:
# Example: Calculate what percentage of total sales each outlet contributes
grand_total = final_report.agg(sum("Total Sales")).collect()[0][0]

sales_pct_example = (final_report
    .withColumn("Outlet_Sales_Percentage_of_Total",
        round(col("Total Sales") / grand_total * 100, 2)
    )
    .select("Outlet_Identifier", "Total Sales", "Outlet_Sales_Percentage_of_Total")
    .orderBy(col("Total Sales").desc())
)

print(f"Grand Total Sales across all outlets: {grand_total}")
sales_pct_example.display()

In [0]:
# ─────────────────────────────────────────────────────────────
# INTERVIEW PRACTICE — Window Functions with Partitions
# (A very common Data Engineer Level 2 interview topic)
# ─────────────────────────────────────────────────────────────

# Q: For each outlet, show a running cumulative total of sales
#    ordered by Item_Type, along with a 3-item-type moving average.

# 1) Build per-outlet, per-ItemType sales so we have rows to rank
item_type_sales = (
    df.groupBy("Outlet_Identifier", "Item_Type")
      .agg(round(sum("Item_Outlet_Sales"), 2).alias("Type_Sales"))
      .orderBy("Outlet_Identifier", "Item_Type")
)

# 2) Define windows partitioned by outlet, ordered by sales
running_window = (
    Window.partitionBy("Outlet_Identifier")
          .orderBy("Item_Type")
          .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

moving_avg_window = (
    Window.partitionBy("Outlet_Identifier")
          .orderBy("Item_Type")
          .rowsBetween(-2, Window.currentRow)          # last 3 rows incl. current
)

prev_window = (
    Window.partitionBy("Outlet_Identifier")
          .orderBy("Item_Type")
)

# 3) Apply: cumulative total, 3-row moving average, and YoY-style delta
window_demo = (
    item_type_sales
        .withColumn("Cumulative_Sales", round(sum("Type_Sales").over(running_window), 2))
        .withColumn("Moving_Avg_3", round(avg("Type_Sales").over(moving_avg_window), 2))
        .withColumn("Prev_Type_Sales", lag("Type_Sales", 1).over(prev_window))
        .withColumn("Sales_Delta", round(col("Type_Sales") - col("Prev_Type_Sales"), 2))
        .withColumn("Growth_Pct",
                    when(col("Prev_Type_Sales").isNull(), lit(None))
                    .otherwise(round((col("Type_Sales") - col("Prev_Type_Sales"))
                              / col("Prev_Type_Sales") * 100, 2)))
)

print("Running totals, moving averages & deltas per outlet:")
window_demo.display()

In [0]:
# ─────────────────────────────────────────────────────────────
#  MINI ETL PIPELINE  (Extract → Transform → Load)
# ─────────────────────────────────────────────────────────────

# 1. EXTRACT — read raw CSV from the volume
raw_df = (
    spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load("/Volumes/workspace/default/my_volume/BigMart Sales.csv")
)
print(f"Extracted {raw_df.count()} raw rows")

# 2. TRANSFORM — clean & enrich
transformed_df = (
    raw_df
        .dropDuplicates()
        .fillna({"Item_Weight": 0, "Outlet_Size": "Unknown"})
        # normalise fat-content labels
        .withColumn("Item_Fat_Content",
            when(col("Item_Fat_Content").isin("LF", "low fat"), "Low Fat")
            .when(col("Item_Fat_Content").isin("reg"), "Regular")
            .otherwise(col("Item_Fat_Content")))
        # derived column: sales per unit MRP
        .withColumn("Sales_Per_MRP",
            round(col("Item_Outlet_Sales") / col("Item_MRP"), 2))
        # drop rows with null target
        .na.drop(subset=["Item_Identifier"])
        .select("Item_Identifier","Item_MRP","Item_Weight", "Outlet_Identifier", "Outlet_Type")
        .filter(col("Item_Weight") >= 15)
        .orderBy(desc("Item_Weight"))
)
print(f"Transformed to {transformed_df.count()} clean rows")

# 3. LOAD — write to a CSV file in the volume (overwrite for idempotent re-runs)
output_path = "/Volumes/workspace/default/my_volume/bigmart_sales_clean.csv"
(transformed_df.write
    .format("csv")
    .mode("overwrite")
    .option("header", "true")
    .save(output_path))
print(f"Saved CSV to: {output_path}")

# quick sanity check
spark.read.format("csv").option("header", "true").load(output_path).display()